## Phage Exploratory Data Analysis

### Segment into Kmers & Heatmap
Done using manipulations.py > fasta_to_kmerdf

In [ ]:
import pandas as pd
from Bio import SeqIO
import numpy as np
import os, sys
from tqdm import tqdm

raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"
scripts_path = "../scripts/"

sys.path.append(scripts_path)
print(sys.path)

from manipulations import fasta_to_kmerdf


In [ ]:
K = 8
phage_kmer_df = fasta_to_kmerdf(raw_data_path+"phagehost_KU/phage_cleaned.fasta", k=K)
display(phage_kmer_df.head())
phage_kmer_counts_df = fasta_to_kmerdf(raw_data_path+"phagehost_KU/phage_cleaned.fasta", k=K, relative=False)


In [ ]:
records = list(SeqIO.parse(raw_data_path + "data2_phages.fasta", "fasta"))
print(len(records))

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from plotly.subplots import make_subplots

# Interactive heatmap with plotly, with barplot of row sums on the right

import plotly.graph_objects as go

# Create heatmap
heatmap = go.Heatmap(
    z=phage_kmer_df.values*100,  # Scale to percentage
    x=phage_kmer_df.columns,
    y=phage_kmer_df.index,
    colorscale="Turbo",
    colorbar=dict(title=f"Relative {K}mer Frequency (%)"),
    showscale=True
)

# Create barplot for row sums
barplot = go.Bar(
    x=phage_kmer_counts_df.sum(axis=1).values,
    y=phage_kmer_counts_df.index,
    orientation='h',
    marker=dict(color='gray'),
    showlegend=False,
    xaxis='x2',
    yaxis='y2'
)

# Create subplots
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.85, 0.15],
    shared_yaxes=True,
    horizontal_spacing=0.00,
    subplot_titles=[f"{K}mer Frequency Heatmap for Phage Sequences", f"{K}mer Counts"]
)

fig.add_trace(heatmap, row=1, col=1)
fig.add_trace(barplot, row=1, col=2)

fig.update_xaxes(title_text=f"{K}mers (Total: {phage_kmer_df.shape[1]})", showticklabels=False, row=1, col=1)
fig.update_xaxes(title_text="Counts", row=1, col=2)
fig.update_yaxes(title_text=f"Phage Sequences (Total: {phage_kmer_df.shape[0]})", row=1, col=1)

fig.update_layout(
    width=1200,
    height=600,
    title_text=f"Distribution of {K}mers Across Phage Sequences, with {K}mer Counts",
    showlegend=False
)

fig.show()

In [ ]:
#Static heatmap with seaborn
plt.figure(figsize=(12,8))
sns.heatmap(phage_kmer_df, cmap="rocket_r")
plt.xlabel(f"{K}mers (Total: {phage_kmer_df.shape[1]})")
plt.ylabel(f"Phage Sequences (Total: {phage_kmer_df.shape[0]})")
plt.title("Kmer Frequency Heatmap for Phage Sequences")

### PCA of high contributing kmers

In [ ]:
import numpy as np
import pandas as pd
from analysis import perform_pca, pca_biplot

phage_labels = [name.split("_")[-1] for name in phage_kmer_df.index.tolist()] # Phage names only
bacteria_hosts = [name.split("_")[0] for name in phage_kmer_df.index.tolist()] # Host bacteria and phage names
features = phage_kmer_df.columns.tolist()  # Kmer features

#### Define PCA first then call biplot - controls pca externally (eg. number of components)
#pca, components, coeff = perform_pca(phage_kmer_df, n_components=2)
#pca_biplot(components, pca.components_.T, PCA = pca, labels = features, n_features_to_plot=10, hide_labels=[True, False])

#### Directly call pca_biplot which performs PCA internally
pca_biplot(data = phage_kmer_df,
            vec_labels = features, point_labels = phage_labels, n_features_to_plot=10, hide_labels=[True, False], color_on=bacteria_hosts)

pca_biplot(data = phage_kmer_df,
           vec_labels = features, point_labels = phage_labels, n_features_to_plot=10, hide_labels=[False, True])


It seems that the phages can be divided into 3 main groupings

In [ ]:
phage_kmer_df.index.tolist()

In [ ]:
phage_pca_clusters = {
    "C1" : ["Pectobacterium_phage_Koroua", "Pectobacterium_phage_Ymer", "Pectobacterium_phage_Amona", "Pectobacterium_phage_Poppous", "Pectobacterium_phage_Taid"],
    "C2" : ["Pectobacterium_phage_Vims", "Lelliottia_phage_Zann", "Pectobacterium_phage_Sabo", "Pectobacterium_phage_Abuela", "Pectobacterium_phage_Guf", "Pectobacterium_phage_Magnum"]

}

### Clustering strains

In [ ]:
n = 500
k = 12
input_phage_path = f"SM_sketches/PhageMinhash_n{n}_k{k}_rev/" 
input_bact_path = f"SM_sketches/BactMinhash_n{n}_k{k}_rev/" 

import sourmash
from Bio import SeqIO
from tqdm import tqdm

records = list(SeqIO.parse(raw_data_path+"phagehost_KU/phage_cleaned.fasta", "fasta"))

#Constructing minhashes for all records
minhashes = []
for rec in tqdm(records, desc="Constructing minhashes for all records", unit="seq"):
    #print("Record:", rec.id, len(rec.seq))
    mh = sourmash.MinHash(n=n, ksize=k) #each record gets its own minhash
    for i in range(0, len(rec.seq) - k + 1):
        kmer = str(rec.seq[i:i+k])
        mh.add_sequence(kmer, force=True)
    minhashes.append(mh)

#Comparing all minhashes
similarity_matrix = dict()
for i, e in enumerate(minhashes):
    sim_inner = dict()
    for j, e2 in enumerate(minhashes):
        x = e.jaccard(minhashes[j])
        sim_inner[records[j].id.split("_")[-1]] = x
    similarity_matrix[records[i].id.split("_")[-1]] = sim_inner

In [ ]:
import seaborn as sns
sns.clustermap(pd.DataFrame(similarity_matrix).T, cmap="viridis", figsize=(10, 10))

In [ ]:
sim_df = pd.DataFrame(similarity_matrix)
sim_df.head()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
cluster_sim = 0.95

# We convert similarity to 'distance' (Distance = 1 - Similarity)
df_dist = 1 - sim_df

# 2. Initialize the clusterer
# 'distance_threshold' is your similarity cutoff
# 'n_clusters=None' is required when using a distance threshold
model = AgglomerativeClustering(
    metric='precomputed', 
    linkage='average', 
    distance_threshold=1 - cluster_sim,
    n_clusters=None
)

# 3. Fit and get labels
cluster_labels = model.fit_predict(df_dist)

# 4. Add labels back to your index
results = pd.Series(cluster_labels, index=sim_df.index, name='Cluster')
print(results.sort_values())